In [1]:
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
import re
import unicodedata
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)

# HTML-Datei auswählen
ordner = Path(r"original_decisions_10000")


dateien = list(ordner.glob("*.html"))

print(len(dateien))

results=[]

for datei in dateien:
    # HTML einlesen
    html = datei.read_text(
        encoding="utf-8",
        errors="replace"
    )
    
    # HTML parsen
    soup = BeautifulSoup(html, "lxml")
    
#===================================== offical Label of decision ==========================================
    aktenzeichen = soup.find("b").get_text(strip=True)

#===================================== First Nummer in Decision-Name ======================================

    dateinamen_ID = int(re.search(r"urteil__(\d{1,5})__", datei.name).group(1))



#===================================== Find language ======================================================
    urteilsbereich = soup.select_one("#highlight_content")
    text = urteilsbereich.get_text("\n", strip=True)
    sprache = None
    if "Besetzung" in text:
        sprache = "de"

    elif "Composition" in text:
        sprache = "fr"

    elif "Composizione" in text:
        sprache = "it"  



#===================================== Besetzungsblock ======================================================

    
    besetzung_block = None
    treffer = None
    
    if sprache == "de":
        treffer = re.search(
            r"(Bundesrichter(?:in)?\s+.+?,\s*"
            r"(?:Präsident|Präsidentin),?)\s+"
            r"Besetzung\s+(.*?)\s+Verfahrensbeteiligte",
            text,
            re.DOTALL | re.IGNORECASE
        )
    
        if not treffer:
            treffer = re.search(
                r"Besetzung\s+(.*?)\s+Verfahrensbeteiligte",
                text,
                re.DOTALL
            )
        
    elif sprache == "fr":
        treffer = re.search(
            r"Composition\s+(.*?)\s+"
            r"(?:Participant(?:e)?s? à la procédure|"
            r"\d+[A-Z]?_\d+/\d{4})",
            text,
            re.DOTALL
        )
    
    elif sprache == "it":
        treffer = re.search(
            r"Composizione\s+(.*?)\s+Partecipanti al procedimento",
            text,
            re.DOTALL
        )



    praesident = None
    einzelrichter = False
    einzelrichter_name = None
    richtergremium = []
    weitere_richter = []
    
    if treffer:
        if len(treffer.groups()) == 1:
            besetzung_block = treffer.group(1)
        else:
            besetzung_block = treffer.group(1) + " " + treffer.group(2)
    
        besetzung_block = re.sub(r"\s+", " ", besetzung_block).strip()




    


#===================================== Gerichtspräsident ======================================


        
        if sprache == "fr":
            treffer_praesident = re.search(
                r"(?:M\.\s+le\s+Juge\s+fédéral|"
                r"Mme\s+la\s+Juge\s+fédérale|"
                r"MM\.\s+les\s+Juges\s+fédéraux|"
                r"MM\.\s+et\s+Mmes\s+les\s+Juges\s+fédéraux|"
    
                r"MM\s+et\s+Mme\s+les\s+Juges\s+fédéraux|"
                r"Mmes\s+et\s+MM\s+les\s+Juges\s+fédéraux|"
                
                r"Mmes\s+et\s+M\.\s+les\s+Juges\s+fédéraux|"
                r"MM\.\s+et\s+Mme\s+les\s+Juges\s+fédéraux|"
                r"MM\.\s+et\s+Mme\s+et\s+les\s+Juges\s+fédéraux)"
                r"\s+(.+?),\s*"
                r"(?:Président|Présidente|Juge présidant|Juge présidante)",
                besetzung_block,
                re.DOTALL | re.IGNORECASE
            )
        
        elif sprache == "de":
            treffer_praesident = re.search(
                r"Bundesrichter(?:in)?\s+(.+?),\s*"
                r"(?:Präsident|Präsidentin|(?:als\s+)?präsidierendes Mitglied)",
                besetzung_block
            )
        
        elif sprache == "it":
            treffer_praesident = re.search(
                r"Giudic[ei]\s+federal[ei]\s+(.+?),\s*"
                r"(?:Presidente|Giudice presidente)",
                besetzung_block,
                re.IGNORECASE
            )
        
        else:
            treffer_praesident = None
        

        if treffer_praesident:
            praesident = treffer_praesident.group(1).strip()


#===================================== weitere Mitglieder des Spruchkörpers ======================================================


        if sprache == "de":
            richter_text = re.split(
                r"Gerichtsschreiber(?:in)?",
                besetzung_block,
                maxsplit=1
            )[0]
        

            richter_text = re.sub(
                r",\s*Präsident(?:in)?\s+Bundesrichter(?:in)?\s+",
                ", ",
                richter_text,
                flags=re.IGNORECASE
            )


            # Zuerst Trennkomma einsetzen
            richter_text = re.sub(
                r"\s+(?=nebenamtliche[rn]?\s+Bundesrichter)",
                ", ",
                richter_text,
                flags=re.IGNORECASE
            )

 

        
            richter_text = re.sub(
                r"(?:nebenamtliche[rn]?\s+)?Bundes?richter(?:in|innen)?\s+",
                "",
                richter_text,
                flags=re.IGNORECASE
            )


            
            rollen = {
                "Präsident",
                "Präsidentin",
                 "Bundesrichter",
                "Bundesrichterin",
                "präsidierendes Mitglied",
                "als präsidierendes Mitglied",
                "als Einzelrichter",
                "als Einzelrichterin",
                "als Instruktionsrichterin",
                "als Instruktionsrichter", 
                "als präsidierendes Miglied",
                "Einzelrichterin",
                "präsisierendes Mitglied",
                

                
            }


            if aktenzeichen == "4A_9/2026":
                print("BESETZUNG:")
                print(repr(besetzung_block))
            
                print("\nRICHTER_TEXT:")
                print(repr(richter_text))
            
               




            
        
        
        elif sprache == "fr":
            richter_text = re.split(
                r"Greffi(?:er|ère)",
                besetzung_block,
                maxsplit=1
            )[0]
        
            richter_text = re.sub(
                r"^(?:"
                r"MM\.?\s+et\s+Mme\s+et\s+les\s+Juges\s+fédéraux,?\s*|"
                r"MM\.?\s+et\s+les\s+Juges\s+fédéraux|"
                r"M\.\s+et\s+Mmes?(?:\s+et)?\s+les\s+Juge?s?\s+fédéraux,?\s*|"
                r"M\.\s+et\s+Mmes\s+les\s+Juges\s+fédéraux|"
                r"MM\.?\s+et\s+Mmes\s+les\s+Juges\s+fédéraux|"
                r"Mmes\s+et\s+M\.\s+les\s+Juge?s?\s+fédéra(?:ux|aux),?\s*|"
                r"Mme\s+et\s+(?:M\.\s+)?les\s+Juge?s?\s+fédéraux,?\s*|"
                r"MM\.?\s+et\s+Mme\s+les\s+Juge?s?\s+fédéraux,?\s*|"
                r"MM\.?\s+les\s+Juge?s?\s+fédéraux,?\s*|"
                r"Mmes?\s+et\s+MM\.?\s+les\s+Juge?s?\s+fédéraux,?\s*|"
                r"Mmes?\s+et\s+MM\.\s+les\s+Juges\s+fédéraux|"
                r"Mmes\s+et\s+M\.\s+les\s+Juges\s+fédéraux|"
                r"MM\.\s+et\s+Mme\s+les\s+Juges\s+fédéraux|"
                r"Mmes\s+les\s+Juges\s+fédéral(?:es|aux)|"
                r"MM\.?\s+les\s+Juges\s+fédéraux|"
                r"Mme\s+les?\s+Juges?\s+fédéraux|"
                
                r"Mme\s+la\s+Juge\s+fédérale|"
                r"M\.\s+le\s+Juge\s+fédéral|"
                r"les\s+Juges\s+fédéraux"
                r")\s+",
                "",
                richter_text,
                flags=re.IGNORECASE
            )
        
            # Beispiel: "Herrmann et Josi" → "Herrmann, Josi"
            richter_text = re.sub(
                r",?\s*(?:présidente?|juge\s+présidant|juge\s+suppléante?)\s*,?",
                ", ", richter_text,flags=re.IGNORECASE)

            richter_text = re.sub(r"\s+et\s+", ", ", richter_text)
        
            rollen = {
                "Président",
                "Présidente",
                "Juge présidant",
                "Juge présidante",
                "en qualité de juge unique",
                "en qualité de juge instructrice",
                "en qualité de Juge unique",
                "en qualité de Juge instructeur",
                "Juge unique",
                "en qualité de",
                "MM",
                "Juge instructrice",
    
            }

            
            
        
        
        elif sprache == "it":
            richter_text = re.split(
                r"Cancellier(?:e|a)",
                besetzung_block,
                maxsplit=1
            )[0]
        
            richter_text = re.sub(
                r"Giudic(?:e|i)\s+federal(?:e|i)\s+",
                "",
                richter_text
            )

            richter_text = re.sub(
                r"\s+e\s+",
                ", ",
                richter_text,
                flags=re.IGNORECASE
            )



            

            richter_text = re.sub(
                r",?\s*(?:Giudice\s+Presidente|Giudice\s+supplente)\s*,?",
                ", ",
                richter_text,
                flags=re.IGNORECASE
            )
      
        
            rollen = {
                "Presidente",
                "Giudice presidente",
                "Giudice unico",
                "in qualità di giudice unico",
                "in qualità di giudice unica"
                
            }


        
        
        

     
        
        richtergremium = [
            re.sub(
                r"^(?:"
                r"MM\.?\s+et\s+Mme\s+et\s+les\s+Juges\s+fédéraux|"
                r"Mmes?\s+les\s+Juges\s+fédéraux|"
                r"MM\.?\s+les\s+Juges\s+fédéraux|"
                r"Mme\s+les?\s+Juges?\s+fédéraux|"
                r"Mme\.?\s+les?\s+Juges?\s+fédérales?|"
                r"Mme?\.?\s+la\s+Juge\s+fédérale|"
                r"(?:M\.\s+la\s+Juge|MM\.?\s+et\s+Mme\s+les\s+Juges)\s+fédérales?|"
                r"M\.\s+le\s+Juge(?:\s+fédéral)?|"
                r"nebenamtlicher\s+Bundesrichter|"
                r"Bundesrichter(?:in)?|"
                r"Bundesricher(?:in)?|"
                r"Bundesricherin|"          
                r"Bundesrichtger|"
                r"Bundesricherin|" 
                r"Bunesrichter|" 
                r"Bundesricherin|"         
                r"Bundsrichter(?:in)?|"
                r"Juge\s+instruct(?:eur|rice)|"
                r"Mme\.?"
                r")\s+",
                "",
                teil.strip(" ."),
                flags=re.IGNORECASE
            )
            for teil in richter_text.split(",")
            if teil.strip(" .")
            and teil.strip(" .") not in rollen
        ]

        richtergremium = [
            re.sub(
                r"^(?:als\s+)?präsidierendes\s+Mitglied\s+|"
                r"\s+als\s+präsidierendes\s+Mitglied$",
                "",
                name,
                flags=re.IGNORECASE
            ).strip()
            for name in richtergremium
        ]





        


        richtergremium = [
            unicodedata.normalize("NFC", name)
            .replace("\u00a0", " ")
            .replace("\u200b", "")
            .strip(" .;,")
            for name in richtergremium
        ]

        richtergremium = [
            name
            for name in richtergremium
            if name
        ]


        

        korrekturen = {
        "Petrik": "Petrik-Haltiner", 
        "Hermann": "Herrmann",
        "Muschetti": "Muschietti",
        "van de Graaaf": "van de Graaf",
        "Van de Graaf": "van de Graaf",
        "Hoffmann": "Hofmann",
        "H ofmann": "Hofmann",
        "Von Felten": "von Felten",
        "de Rossa": "De Rossa",
        "Koc h": "Koch",
         "Kradofler": "Kradolfer",
        "Pont Venthey": "Pont Veuthey",
        "Kneuhühler": "Kneubühler",
        "Müller Th": "Müller",
        "et Hofmann": "Hofmann",
         "Präsident van de Graaf": "van de Graaf",
        "Präsident Abrecht": "Abrecht",
        "les Juges fédéraux Abrecht": "Abrecht",
        "MM. Juges fédéraux Abrecht": "Abrecht",
        "M. Mmes les Juges fédéraux Abrecht": "Abrecht",
        "M. les Juge fédéral Merz": "Merz",
        "M. Donzallaz": "Donzallaz",

      
       
        
            # nur falls wir nach Prüfung sehen, dass das immer stimmt
        }

        richtergremium = [
            korrekturen.get(name, name) 
            for name in richtergremium
        ]

        praesident = korrekturen.get(praesident, praesident)

        
        
        weitere_richter = [
            name
            for name in richtergremium
            if name != praesident
        ]
        
        if len(richtergremium) == 1:
            einzelrichter = True
            einzelrichter_name = richtergremium[0]


#===================================== einzelrichter ======================================================

  

        if sprache == "de":
            treffer_einzelrichter = re.search(
                r"Bundesrichter(?:in)?\s+(.+?),\s+als Einzelrichter(?:in)?",
                besetzung_block
            )
        
        elif sprache == "fr":
            treffer_einzelrichter = re.search(
                r"(?:M\.|Mme)\s+le Juge fédéral\s+(.+?),\s+"
                r"en qualité de juge unique",
                besetzung_block,
                re.IGNORECASE
            )
        
        elif sprache == "it":
            treffer_einzelrichter = re.search(
                r"Giudice federale\s+(.+?),\s+"
                r"in qualità di giudice unic[oa]",
                besetzung_block,
                re.IGNORECASE
            )
        
        else:
            treffer_einzelrichter = None
        
        if treffer_einzelrichter:
            einzelrichter = True
            einzelrichter_name = treffer_einzelrichter.group(1).strip()

        

 
#===================================== Make dictionary ======================================================
    resultat = {
        "id_dateinamen": dateinamen_ID,
        "dateiname": datei.name,
        "aktenzeichen": aktenzeichen,
        "sprache": sprache,
        "besetzung_block": besetzung_block,
        "praesident": praesident,
        "einzelrichter": einzelrichter,
        "einzelrichter_name": einzelrichter_name,
        "weiterer_richter1": weitere_richter[0] if len(weitere_richter) > 0 else None,
        "weiterer_richter2": weitere_richter[1] if len(weitere_richter) > 1 else None,
         "weiterer_richter3": weitere_richter[2] if len(weitere_richter) > 2 else None,
        "weiterer_richter4": weitere_richter[3] if len(weitere_richter) > 3 else None,
        
        
        }

    results.append(resultat)
    
    
# DataFrame erstellen
df = pd.DataFrame(results)  



10000
BESETZUNG:
'Bundesrichter Hurni, Präsident Bundesrichterin Kiss, Bundesrichter Denys, Rüedi, Bundesrichterin May Canellas, Gerichtsschreiber Tanner.'

RICHTER_TEXT:
'Hurni, Kiss, Denys, Rüedi, May Canellas, '


In [2]:
df[["id_dateinamen","praesident","weiterer_richter1","weiterer_richter2","besetzung_block"]].head(3).fillna("")

,id_dateinamen,praesident,weiterer_richter1,weiterer_richter2,besetzung_block
0,10000,Moser-Szeless,Stadelmann,Parrino,"Mmes et MM. les Juges fédéraux Moser-Szeless, Présidente, Stadelmann, Parrino, Beusch et Bollinger. Greffier : M. Feller."
1,1000,Bovey,,,"Bundesrichter Bovey, Präsident, Gerichtsschreiber Zingg."
2,1001,Aubry Girardin,Donzallaz,Hänni,"Bundesrichterin Aubry Girardin, Präsidentin, Bundesrichter Donzallaz, Bundesrichterin Hänni, Bundesrichterin Ryter, Bundesrichter Kradolfer, Gerichtsschreiber Kaufmann."


In [3]:
df[df["sprache"] == "fr"].head(3)

,id_dateinamen,dateiname,aktenzeichen,sprache,besetzung_block,praesident,einzelrichter,einzelrichter_name,weiterer_richter1,weiterer_richter2,weiterer_richter3,weiterer_richter4
0,10000,urteil__10000__16_2025_03_21__9C_75-2024.html,9C_75/2024,fr,"Mmes et MM. les Juges fédéraux Moser-Szeless, Présidente, Stadelmann, Parrino, Beusch et Bollinger. Greffier : M. Feller.",Moser-Szeless,False,NaN,Stadelmann,Parrino,Beusch,Bollinger
4,1003,urteil__1003__3_2026_05_28__1C_288-2025.html,1C_288/2025,fr,"M. le Juge fédéral Chaix, en qualité de Juge instructeur. Greffière : Mme Rouiller.",NaN,True,Chaix,Chaix,NaN,NaN,NaN
6,1005,urteil__1005__5_2026_05_28__1C_174-2025.html,1C_174/2025,fr,"MM. les Juges fédéraux Haag, Président, Kneubühler et Merz. Greffier : M. Alvarez.",Haag,False,NaN,Kneubühler,Merz,NaN,NaN


In [4]:
df.loc[
    (df["sprache"] == "de") &
    (df["weiterer_richter1"].isna()),
    [
        "id_dateinamen",
        "praesident",
        "einzelrichter",
        "einzelrichter_name",
        "besetzung_block"
    ]
].head(3)

,id_dateinamen,praesident,einzelrichter,einzelrichter_name,besetzung_block
1,1000,Bovey,True,Bovey,"Bundesrichter Bovey, Präsident, Gerichtsschreiber Zingg."
10,1009,Haag,True,Haag,"Bundesrichter Haag, Präsident, Gerichtsschreiber Baur."
17,1015,Haag,True,Haag,"Bundesrichter Haag, Präsident, Gerichtsschreiber Baur."


In [5]:
df_decision_judge = df.melt(
    id_vars=[
        "id_dateinamen",
        "aktenzeichen"
    ],
    value_vars=[
        "praesident",
        "weiterer_richter1",
        "weiterer_richter2",
        "weiterer_richter3",
        "weiterer_richter4",
    ],
    var_name="funktion",
    value_name="nachname"
)


In [6]:
df_decision_judge.head(3)

,id_dateinamen,aktenzeichen,funktion,nachname
0,10000,9C_75/2024,praesident,Moser-Szeless
1,1000,5A_449/2026,praesident,Bovey
2,1001,2E_8/2024,praesident,Aubry Girardin


In [7]:
df_richter = pd.read_excel("bundesrichter_ab_2007.xlsx")

In [8]:
df_richter.head(3)

,Name,Wahl,Rücktritt,geboren,gestorben,Kanton,party,nachname,Partei
0,Arthur Brunner,2026.0,NaN,NaN,NaN,NaN,SVP,Brunner,SVP
1,Bernard Abrecht,2019.0,NaN,1966.0,NaN,Waadt / Bern,SP,Abrecht,SP
2,Florence Aubry Girardin,2007.0,NaN,1964.0,NaN,Jura,Grüne,Aubry Girardin,Grüne


In [9]:
df_merged = df_decision_judge.merge(
    df_richter,
    on="nachname",
    how="left"
)



In [10]:
df_merged = df_merged.drop(columns="party")
df_merged.head(2)

,id_dateinamen,aktenzeichen,funktion,nachname,Name,Wahl,Rücktritt,geboren,gestorben,Kanton,Partei
0,10000,9C_75/2024,praesident,Moser-Szeless,Margit Moser-Szeless,2014.0,NaN,1971.0,NaN,Genf / Luzern,SVP
1,1000,5A_449/2026,praesident,Bovey,Grégory Bovey,2014.0,NaN,1973.0,NaN,Waadt,FDP


In [11]:
#Test: Does every last name in the merged DataFrame have an exact match in the judges DataFrame?
k = 0

for richter in df_merged["nachname"]:

    if pd.isna(richter):
        continue

    n = 0

    for prof in df_decision_judge["nachname"]:
        if prof == richter:
            #print(richter + " VS " +prof)
            n = 1
            break

    if n == 0:
        print("Kein Treffer:", richter)
        k = k + 1

print("Anzahl ohne Treffer:", k)

Anzahl ohne Treffer: 0


In [12]:
df_merged[df_merged["Name"].isna()]["nachname"].value_counts()

Series([], Name: count, dtype: int64)

In [13]:
df_merged[
    df_merged["Name"].isna()
]["nachname"].isna().sum()


np.int64(26497)

In [14]:
df_merged[
    df_merged["Name"].isna()
    & df_merged["nachname"].str.contains(" ", na=False)
]["nachname"].value_counts().sum()

np.int64(0)

In [15]:
#Test: decicions only with 1,3 or 5 Judges

Number_with_1 = 0
Number_with_2 = 0
Number_with_3 = 0
Number_with_4 = 0
Number_with_5 = 0

problem_ids = []

for id_dateinamen, urteil in df_merged[df_merged["nachname"].notna()].groupby("id_dateinamen").size().items():
    if urteil == 1:
        Number_with_1 += 1
    elif urteil == 2:
        Number_with_2 += 1
    elif urteil == 3:
        Number_with_3 += 1
    elif urteil == 4:
        problem_ids.append(id_dateinamen)
        Number_with_4 += 1
    elif urteil == 5:
        Number_with_5 += 1

print(problem_ids)

print("Number of decisions with 1 judge: " + str(Number_with_1))
print("Number of decisions with 2 judges: " + str(Number_with_2))
print("Number of decisions with 3 judges: " + str(Number_with_3))
print("Number of decisions with 4 judges: " + str(Number_with_4))
print("Number of decisions with 5 judges: " + str(Number_with_5))
        

[]
Number of decisions with 1 judge: 3861
Number of decisions with 2 judges: 0
Number of decisions with 3 judges: 5504
Number of decisions with 4 judges: 0
Number of decisions with 5 judges: 626


In [16]:
richter_pro_urteil = (
    df_merged[df_merged["nachname"].notna()]
    .groupby("id_dateinamen")
    .size()
)

ids_mit_4 = richter_pro_urteil[richter_pro_urteil == 2].index

df_4er = df_merged[
    df_merged["id_dateinamen"].isin(ids_mit_4)
][
    ["id_dateinamen", "aktenzeichen", "funktion", "nachname"]
].sort_values(["id_dateinamen", "funktion"])

In [17]:
df_4er.head(40)

,id_dateinamen,aktenzeichen,funktion,nachname


In [30]:
df_merged.groupby("Partei").size()

Partei
Die Mitte             3277
FDP                   4252
GLP                   1842
Grüne                 2706
Mitte                  536
SP                    5105
SVP                   5278
parteilos (ex SVP)     507
dtype: int64

In [34]:
#Decisions with 3 Judges

urteile_3 = (
    df_merged
    .groupby("aktenzeichen")["nachname"]
    .count()
    .loc[lambda s: s == 3]
)

df_merged[
    df_merged["aktenzeichen"].isin(urteile_3.index)
].sort_values(["aktenzeichen", "funktion"])

,id_dateinamen,aktenzeichen,funktion,nachname,Name,Wahl,Rücktritt,geboren,gestorben,Kanton,Partei
4962,5467,11Z_1/2025,praesident,Bovey,Grégory Bovey,2014.0,NaN,1973.0,NaN,Waadt,FDP
14962,5467,11Z_1/2025,weiterer_richter1,Hartmann,Stephan Hartmann,2021.0,NaN,1972.0,NaN,Aargau,Grüne
24962,5467,11Z_1/2025,weiterer_richter2,Josi,Christian Josi,2024.0,NaN,1973.0,NaN,Bern,SVP
34962,5467,11Z_1/2025,weiterer_richter3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
44962,5467,11Z_1/2025,weiterer_richter4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2550,3296,9G_3/2025,praesident,Moser-Szeless,Margit Moser-Szeless,2014.0,NaN,1971.0,NaN,Genf / Luzern,SVP
12550,3296,9G_3/2025,weiterer_richter1,Parrino,Francesco Parrino,2013.0,NaN,1967.0,NaN,Tessin,SP
22550,3296,9G_3/2025,weiterer_richter2,Bollinger,Susanne Bollinger,2024.0,NaN,1974.0,NaN,Schaffhausen,SVP
32550,3296,9G_3/2025,weiterer_richter3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
#Three-judge panels: Decisions with at least two right-of-center(SVP And FDP) judges
df_3 = df_merged[
    df_merged["aktenzeichen"].isin(urteile_3.index)
]
urteile_svp_fdp_3 = (
    df_3
    .groupby("aktenzeichen")["Partei"]
    .agg(lambda x: {"FDP", "SVP"}.issubset(set(x.dropna())))
)
urteile_svp_fdp_3.sum()

np.int64(1469)

In [46]:
#Three-judge panels: Decisions with at least two right-of-center(SVP And FDP) judges
df_3 = df_merged[
    df_merged["aktenzeichen"].isin(urteile_3.index)
]
urteile_svp_fdp_3 = (
    df_3
    .groupby("aktenzeichen")["Partei"]
    .agg(lambda x: {"SP", "Grüne"}.issubset(set(x.dropna())))
)
urteile_svp_fdp_3.sum()

np.int64(853)